# 3D Packing Problem

#### Importing packages

In [107]:
import numpy as np 
import pandas as pd
from amplpy import AMPL

### Notations

- $I=\{1,2,...,N\}$ is the set of SKUs, where N denotes the total number of SKUs to be packed.
  
- $t$  is the number of copies of the DFC

- $T$ is the set of copies of the DFC

- The $i^{th}$ SKU is characterized by its length $l_i$, width $w_i$, height $h_i$, weight $m_i$. $\forall i \in I$

- The DFC is characterized by its length $L$, width $W$, height $H$ and maximum weight carrying capacity $G$

- $lr^+_{ii′} = 1$, if $i^{th}$ SKU is placed on left-hand side of $i′^{th}$ SKU; otherwise, $lr^+_{ii′} = 0$.
- $bf^+_{ii′} = 1$, if $i^{th}$ SKU is placed behind the $i′^{th}$ SKU;    otherwise, $bf^+_{ii′} = 0$.
- $bt^+_{ii′} = 1$, if $i^{th}$ SKU is placed on below the $i′^{th}$ SKU; otherwise, $bt^+_{ii′} = 0$.
- $lr^-_{ii′} = 1$, if $i^{th}$ SKU is placed on right-hand side of $i′^{th}$  SKU; otherwise, $lr^-_{ii′} = 0$.
- $bf^-_{ii′} = 1$, if $i^{th}$ SKU is placed in front of $i′^{th}$ SKU; otherwise, $bf^-_{ii′} = 0$.
- $bt^-_{ii′} = 1$, if $i^{th}$ SKU is placed above the $i′^{th}$ SKU; otherwise, $bt^-_{ii′} = 0$.

- $u_k$ is a binary variable, which denotes whether $k_{th}$ copy of the DFC.


- $s_{ik}$ is a binary variable to denote, whether the $i_{th}$ SKU is packed in $k_{th}$ copy of
the DFC.

- The variables $x_i$, $y_i$, $z_i$ denotes the coodinated of the left-bottom-back corner of
$i_{th}$ SKU.

- The Variables $l^x_i$ , $l^y_i$ , $l^z_i$ denotes whether the length edge of ith SKU is parallel to X-axis, Y-axis or Z-axis.

- Similarly, the variables $w^x_i$ , $w^y_i$ , $w^z_i$ denotes whether the width edge of ith SKU is parallel to X-axis, Y-axis or Z-axis, and the variables $h^x_i$ , $h^y_i$ , $h^z_i$ denotes whether the height edge of $i_{th}$ SKU is parallel to X-axis, Y-axis or Z-axis.


### Formulation

The objective is to minimize the total vacant spaces or unused volume within
the assigned DFC or, equivalently, the total volume of the chosen DFC for
packing. The latter is easier to model. The overall model is as follows: 

$$
\begin{align}

\min \quad 
& \sum_{k\in T}  u_k L W H
- \sum_{i\in I} l_i w_i h_i \\

\text{s.t.} \quad 
& \sum_{k\in T} s_{ik} = 1 
&& \forall i \in I \\

& u_k \ge s_{ik}
&& \forall i \in I,\; \forall k \in T_j \\

& l_i^x + l_i^y + l_i^z = 1
&& \forall i \in I \\

& w_i^x + w_i^y + w_i^z = 1
&& \forall i \in I \\

& h_i^x + h_i^y + h_i^z = 1
&& \forall i \in I \\

& l_i^x + w_i^x + h_i^x = 1
&& \forall i \in I \\

& l_i^y + w_i^y + h_i^y = 1
&& \forall i \in I \\

& l_i^z + w_i^z + h_i^z = 1
&& \forall i \in I \\

& x_i + l_i^x l_i + w_i^x w_i + h_i^x h_i
\le L + (1-s_{ik})M
&& \forall i\in I,\forall k\in T \\

& y_i + l_i^y l_i + w_i^y w_i + h_i^y h_i
\le W + (1-s_{ik})M
&& \forall i\in I,\forall k\in T \\

& z_i + l_i^z l_i + w_i^z w_i + h_i^z h_i
\le H + (1-s_{ik})M
&& \forall i\in I,\forall k\in T \\

& \sum_{i\in I} s_{ik} m_i \le G
&& \forall k\in T \\

& x_i + l_i^x l_i + w_i^x w_i + h_i^x h_i
\le x_{i'} + (1-lr_{ii'}^{+})M
&& i<i',\forall i,i'\in I \\

& x_{i'} + l_{i'}^x l_{i'} + w_{i'}^x w_{i'} + h_{i'}^x h_{i'}
\le x_i + (1-lr_{ii'}^{-})M
&& i<i',\forall i,i'\in I \\

& y_i + l_i^y l_i + w_i^y w_i + h_i^y h_i
\le y_{i'} + (1-bf_{ii'}^{+})M
&& i<i',\forall i,i'\in I \\

& y_{i'} + l_{i'}^y l_{i'} + w_{i'}^y w_{i'} + h_{i'}^y h_{i'}
\le y_i + (1-bf_{ii'}^{-})M
&& i<i',\forall i,i'\in I \\

& z_i + l_i^z l_i + w_i^z w_i + h_i^z h_i
\le z_{i'} + (1-bt_{ii'}^{+})M
&& i<i',\forall i,i'\in I \\

& z_{i'} + l_{i'}^z l_{i'} + w_{i'}^z w_{i'} + h_{i'}^z h_{i'}
\le z_i + (1-bt_{ii'}^{-})M
&& i<i',\forall i,i'\in I \\

& lr_{ii'}^{+} + lr_{ii'}^{-}
+ bf_{ii'}^{+} + bf_{ii'}^{-}
+ bt_{ii'}^{+} + bt_{ii'}^{-}
\ge s_{ik} + s_{i'jk} - 1
&& i<i',\forall i,i'\in I,\forall k\in T\\

& u_p \ge u_q
\qquad
&& \forall
p,q \in T: q = p + 1

\end{align}
$$

#### Make model

In [108]:
def model(a):
    # Data formatting
    with pd.ExcelFile(a) as xls:
        sku_df = pd.read_excel(xls, "SKU")

    print(sku_df.to_string())
    
    # Initialize ampl object
    ampl = AMPL()

    ampl.eval(
        """
        set SKU;                             
        set Copy;                            # set of copies of the DFC with elements equal to the number of SKU
        set axis= {'x', 'y', 'z'};
        set dim= {'Length', 'Width', 'Height'};

        param dim_sku {SKU, dim};
        param sku_Weight {SKU};
        
        param M;       

        var dim_dfc {dim} >=0;
        var dfc_Weight >=0;

        var relative_position_left {i in SKU, l in SKU : i<l} binary;
        var relative_position_right {i in SKU, l in SKU : i<l} binary;
        var relative_position_back {i in SKU, l in SKU : i<l} binary;
        var relative_position_front {i in SKU, l in SKU : i<l} binary;
        var relative_position_below {i in SKU, l in SKU : i<l} binary;
        var relative_position_above {i in SKU, l in SKU : i<l} binary;

        var copy_used {k in Copy} binary;
        var sku_in_copy {i in SKU, k in Copy} binary;
        var sku_position {SKU, axis} >=0;

        var Length_orientation {SKU, axis} binary;
        var Width_orientation {SKU, axis} binary;
        var Height_orientation {SKU, axis} binary;

        minimize vacant_space:
        sum{k in Copy} copy_used[k]*dim_dfc['Length']*dim_dfc['Width']*dim_dfc['Height'] - sum{i in SKU} dim_sku[i,'Length']*dim_sku[i,'Width']*dim_sku[i,'Height']; 

        s.t. c1 {i in SKU}: 
        sum{k in Copy} sku_in_copy[i,k] = 1;

        s.t. c2 {i in SKU}:
        Length_orientation[i,'x'] + Length_orientation[i,'y'] + Length_orientation[i,'z'] = 1;

        s.t. c3 {i in SKU}:
        Width_orientation[i,'x'] + Width_orientation[i,'y'] + Width_orientation[i,'z'] = 1;

        s.t. c4 {i in SKU}:
        Height_orientation[i,'x'] + Height_orientation[i,'y'] + Height_orientation[i,'z'] = 1;

        s.t. c5 {i in SKU}:
        Length_orientation[i,'x'] + Width_orientation[i,'x'] + Height_orientation[i,'x'] = 1;

        s.t. c6 {i in SKU}:
        Length_orientation[i,'y'] + Width_orientation[i,'y'] + Height_orientation[i,'y'] = 1;

        s.t. c7 {i in SKU}:
        Length_orientation[i,'z'] + Width_orientation[i,'z'] + Height_orientation[i,'z'] = 1;

        s.t. c8 {i in SKU, k in Copy}:  
        sku_position[i,'x'] + Length_orientation[i,'x']*dim_sku[i,'Length'] + Width_orientation[i,'x']*dim_sku[i,'Width'] + Height_orientation[i,'x']*dim_sku[i,'Height'] <= dim_dfc[ 'Length'] + (1- sku_in_copy[i,k])*M;
        
        s.t. c9 {i in SKU, k in Copy}: 
        sku_position[i,'y'] + Length_orientation[i,'y']*dim_sku[i,'Length'] + Width_orientation[i,'y']*dim_sku[i,'Width'] + Height_orientation[i,'y']*dim_sku[i,'Height'] <= dim_dfc[ 'Width'] + (1- sku_in_copy[i,k])*M;
        
        s.t. c10 {i in SKU, k in Copy}: 
        sku_position[i,'z'] + Length_orientation[i,'z']*dim_sku[i,'Length'] + Width_orientation[i,'z']*dim_sku[i,'Width'] + Height_orientation[i,'z']*dim_sku[i,'Height'] <= dim_dfc[ 'Height'] + (1- sku_in_copy[i,k])*M;

        s.t. c11 {i in SKU, k in Copy}: 
        copy_used[k] >= sku_in_copy[i,k];

        s.t. c12 {k in Copy}:
        sum{i in SKU} sku_in_copy[i,k]*sku_Weight[i] <= dfc_Weight;

        s.t. c13 {(i,l) in {i in SKU, l in SKU : i<l}}:
        sku_position[i,'x'] + Length_orientation[i,'x']*dim_sku[i,'Length'] + Width_orientation[i,'x']*dim_sku[i,'Width'] + Height_orientation[i,'x']*dim_sku[i,'Height'] <= sku_position[l,'x'] + (1- relative_position_left[i,l])*M;
        
        s.t. c14 {(i,l) in {i in SKU, l in SKU : i<l}}:
        sku_position[l,'x'] + Length_orientation[l,'x']*dim_sku[l,'Length'] + Width_orientation[l,'x']*dim_sku[l,'Width'] + Height_orientation[l,'x']*dim_sku[l,'Height'] <= sku_position[i,'x'] + (1- relative_position_right[i,l])*M;

        s.t. c15 {(i,l) in {i in SKU, l in SKU : i<l}}:
        sku_position[i,'y'] + Length_orientation[i,'y']*dim_sku[i,'Length'] + Width_orientation[i,'y']*dim_sku[i,'Width'] + Height_orientation[i,'y']*dim_sku[i,'Height'] <= sku_position[l,'y'] + (1- relative_position_back[i,l])*M;
        
        s.t. c16 {(i,l) in {i in SKU, l in SKU : i<l}}:
        sku_position[l,'y'] + Length_orientation[l,'y']*dim_sku[l,'Length'] + Width_orientation[l,'y']*dim_sku[l,'Width'] + Height_orientation[l,'y']*dim_sku[l,'Height'] <= sku_position[i,'y'] + (1- relative_position_front[i,l])*M;

        s.t. c17 {(i,l) in {i in SKU, l in SKU : i<l}}:
        sku_position[i,'z'] + Length_orientation[i,'z']*dim_sku[i,'Length'] + Width_orientation[i,'z']*dim_sku[i,'Width'] + Height_orientation[i,'z']*dim_sku[i,'Height'] <= sku_position[l,'z'] + (1- relative_position_below[i,l])*M;
        
        s.t. c18 {(i,l) in {i in SKU, l in SKU : i<l}}:
        sku_position[l,'z'] + Length_orientation[l,'z']*dim_sku[l,'Length'] + Width_orientation[l,'z']*dim_sku[l,'Width'] + Height_orientation[l,'z']*dim_sku[l,'Height'] <= sku_position[i,'z'] + (1- relative_position_above[i,l])*M;
        
        s.t. c19 {i in SKU, l in SKU, k in Copy : i<l}:
        relative_position_left[i,l] + relative_position_right[i,l] + relative_position_back[i,l] + relative_position_front[i,l] + relative_position_below[i,l] + relative_position_above[i,l] >= sku_in_copy[i,k] + sku_in_copy[l,k] -1;

        s.t. c20 {p in Copy, q in Copy : q = p+1}:
        copy_used[p] >= copy_used[q];
        """
        )

    # Feeding data to the model
    ampl.set['SKU'] = sku_df['sku']
    ampl.set['Copy'] = sku_df['sku']    # maximum number of copies of DFC can be equal to the number of SKU at most

    ampl.param['dim_sku'] = sku_df.set_index('sku').loc[:,['Length', 'Width', 'Height']]
    ampl.param['sku_Weight'] = sku_df.set_index('sku')['Weight']
    ampl.param['M'] = float(sku_df[['Length', 'Width', 'Height']].sum().max())

    # Solve 
    ampl.option['solver'] = 'gurobi'
    ampl.option['gurobi_options']= 'outlev=1'
    ampl.option['timelimit'] = 500
    ampl.solve()

    solve_result = ampl.get_value('solve_result')
    print(f"Solve result: {solve_result}")
    print(f"Objective:    {ampl.get_value('vacant_space')}")


    if 'infeasible' in solve_result:
        # --- IIS block: only runs when broken ---
        ampl.option["gurobi_options"] = "iisfind=1"
        ampl.solve()

        from amplpy import OutputHandler

        class CaptureOutput(OutputHandler):
            def __init__(self): self.lines = []
            def output(self, kind, msg): self.lines.append(msg)

        capture = CaptureOutput()
        ampl.set_output_handler(capture)
        ampl.eval("""
            for {j in 1.._ncons} {
                if _con[j].iis != 'non' then
                    printf "CON | %-6s | %s\\n", _con[j].iis, _conname[j];
            }
            for {j in 1.._nvars} {
                if _var[j].iis != 'non' then
                    printf "VAR | %-6s | %s\\n", _var[j].iis, _varname[j];
            }
        """)
        ampl.set_output_handler(None)

        
        records = []
        for line in capture.lines:
            line = line.strip()
            if line.startswith("CON |") or line.startswith("VAR |"):
                parts = [p.strip() for p in line.split("|")]
                records.append({"kind": parts[0], "iis": parts[1], "name": parts[2]})

        if records:
            df = pd.DataFrame(records)
            df["group"] = df["name"].str.replace(r"\[.*\]", "", regex=True)
            print("\n=== IIS Members ===")
            print(df.to_string(index=False))
            print("\n=== IIS by constraint group ===")
            print(df.groupby(["kind","group","iis"]).size()
                    .reset_index(name="count").to_string(index=False))

    else:
        # --- Results block: only runs when solved ---
        dim_dfc_df = ampl.get_variable('dim_dfc').to_pandas()
        dfc_weight_df = ampl.get_variable('dfc_Weight').to_pandas()

        copy_used_df     = ampl.var['copy_used'].get_values().to_pandas()
        sku_in_copy_df   = ampl.var['sku_in_copy'].get_values().to_pandas()
        sku_position_df  = ampl.var['sku_position'].get_values().to_pandas()
        Length_orientation_df = ampl.var['Length_orientation'].get_values().to_pandas()
        Width_orientation_df = ampl.var['Width_orientation'].get_values().to_pandas()
        Height_orientation_df = ampl.var['Height_orientation'].get_values().to_pandas()

        # Scalar counts and volumes
        copies_used    = int((copy_used_df['copy_used.val'] > 0.5).sum())

        # dim_dfc_df has one row with L, W, H columns — product across columns
        dfc_volume_each  = float(dim_dfc_df.prod(axis=0).iloc[0])   # L × W × H
        dfc_volume_total = copies_used * dfc_volume_each

        # vacant_space = total DFC volume used − total SKU volume (your objective)
        # so packing efficiency = 1 - vacant / total_dfc
        packing_efficiency = 1 - ampl.get_value('vacant_space') / dfc_volume_total

        # Gurobi reports its own solve time via this suffix
        gurobi_time = ampl.get_value('_total_solve_time')
        print(f"Solve time (Gurobi): {gurobi_time:.4f} seconds")

        print(f"Packing efficiency: {packing_efficiency * 100:.2f}%")

        print("\n=== DFC dimension ===")        
        print(dim_dfc_df)

        print("\n=== DFC weight capacity ===")
        print(dfc_weight_df)

        print("\n=== Copies used ===")
        print(copy_used_df[copy_used_df['copy_used.val'] > 0.5])

        print("\n=== SKU assignments ===")
        print(sku_in_copy_df[sku_in_copy_df['sku_in_copy.val'] > 0.5])

        print("\n=== SKU positions ===")
        print(sku_position_df)

        print("\n=== Length orientation ===")
        print(Length_orientation_df[Length_orientation_df['Length_orientation.val'] > 0.5])

        print("\n=== Width orientation ===")
        print(Width_orientation_df[Width_orientation_df['Width_orientation.val'] > 0.5])

        print("\n=== Height orientation ===")
        print(Height_orientation_df[Height_orientation_df['Height_orientation.val'] > 0.5])

In [109]:
model("Capacity_prob.xlsx")

   sku  Length  Width  Height  Weight
0    1       2      2       2       2
1    2       1      1       1       1
2    3       2      2       2       2
3    4       1      1       1       1
4    5       2      2       2       2
5    6       2      2       2       2
Gurobi 12.0.3: Set parameter LogToConsole to value 1
  tech:outlev = 1
Set parameter InfUnbdInfo to value 1
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) 5 120U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Non-default parameters:
InfUnbdInfo  1

Optimize a model with 377 rows, 222 columns and 2176 nonzeros
Model fingerprint: 0xd1724cbe
Model has 1 general nonlinear constraint (18 nonlinear terms)
Variable types: 36 continuous, 186 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 3e+01]
  RHS range        

#### Edge cases 
- an sku that can fit in only one particular dfc

In [110]:
model('3dpack1.xlsx')

   sku  Length  Width  Height  Weight
0    1       3      3       3       3
1    2       3      3       3       3
2    3       4      4       3       3
3    4       3      3       3       3
4    5       3      4       4       3
5    6       2      3       3       3
6    7       5      5       5       5
Gurobi 12.0.3: Set parameter LogToConsole to value 1
  tech:outlev = 1
Set parameter InfUnbdInfo to value 1
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) 5 120U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Non-default parameters:
InfUnbdInfo  1

Optimize a model with 531 rows, 286 columns and 3155 nonzeros
Model fingerprint: 0x563d20d8
Model has 1 general nonlinear constraint (21 nonlinear terms)
Variable types: 41 continuous, 245 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range 

- a set of sku which can fit in one particular dfc in one particular orientation

this is not possible as 

In [111]:
model('3dpack2.xlsx')

   sku  Length  Width  Height  Weight
0    1       2      3       4       5
1    2       6      4       5       7
Gurobi 12.0.3: Set parameter LogToConsole to value 1
  tech:outlev = 1
Set parameter InfUnbdInfo to value 1
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) 5 120U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Non-default parameters:
InfUnbdInfo  1

Optimize a model with 41 rows, 46 columns and 180 nonzeros
Model fingerprint: 0xa2e4d72a
Model has 1 general nonlinear constraint (6 nonlinear terms)
Variable types: 16 continuous, 30 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 9e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+02]
  RHS range        [1e+00, 9e+00]
Presolve model has 1 nlconstr
Added 2 variables to disaggregate expressions.
Presolve removed 7 rows and 8 columns
Presolve time: 0.00s
Presol

- an unfitable sku

this is not possible as the dimensions of dfc are variables.


In [112]:
model('3dpack3.xlsx')

   sku  Length  Width  Height  Weight
0    1       2      2       2       2
1    2       1      1       1       1
2    3       2      2       2       2
3    4       1      1       1       1
4    5       2      2       2       2
5    6       2      2       2       2
6    7       4      4       5       7
Gurobi 12.0.3: Set parameter LogToConsole to value 1
  tech:outlev = 1
Set parameter InfUnbdInfo to value 1
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) 5 120U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Non-default parameters:
InfUnbdInfo  1

Optimize a model with 531 rows, 286 columns and 3155 nonzeros
Model fingerprint: 0xd7036696
Model has 1 general nonlinear constraint (21 nonlinear terms)
Variable types: 41 continuous, 245 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range 

- adding one sku occupies one more dfc

this edge case is not possible as trying do so just increases the dfc dimension that's all it does.

In [113]:
model('3dpack4.xlsx')

from io import BytesIO

# Read the original Excel file into memory
with open('3dpack4.xlsx', 'rb') as f:
    file_bytes = BytesIO(f.read())

# Load the sheet you want
df = pd.read_excel(file_bytes, sheet_name='SKU')

# Add a new row
new_row = pd.DataFrame([{'sku': 7, 'Length': 3, 'Width': 3, 'Height': 4, 'Weight': 3}])
df = pd.concat([df, new_row], ignore_index=True)

# Write the modified df back to a BytesIO buffer (never touches disk)
buffer = BytesIO()
with pd.ExcelWriter(buffer, engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='SKU', index=False)
buffer.seek(0)

# Now use that buffer anywhere that accepts a file-like object
model(buffer)

   sku  Length  Width  Height  Weight
0    1       2      2       2       2
1    2       1      1       1       1
2    3       2      2       2       2
3    4       1      1       1       1
4    5       2      2       2       2
5    6       2      2       2       2
Gurobi 12.0.3: Set parameter LogToConsole to value 1
  tech:outlev = 1
Set parameter InfUnbdInfo to value 1
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) 5 120U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Non-default parameters:
InfUnbdInfo  1

Optimize a model with 377 rows, 222 columns and 2176 nonzeros
Model fingerprint: 0xd1724cbe
Model has 1 general nonlinear constraint (18 nonlinear terms)
Variable types: 36 continuous, 186 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 3e+01]
  RHS range        